In [1]:
import json

In [2]:
# ===== 1. WCZYTANIE =====
with open("konstytucja.txt", encoding="utf-8") as f:
    konstytucja = f.read()
print("Korpus:", len(konstytucja), "znaków")

Korpus: 56535 znaków


In [4]:
konstytucja = konstytucja.replace("„", '"')
konstytucja = konstytucja.replace("”", '"')
konstytucja = konstytucja.replace("–", "-")
konstytucja = konstytucja.replace("\n", " ")
while "  " in konstytucja:
    konstytucja = konstytucja.replace("  ", " ")

In [7]:
tokeny = list(konstytucja)   # napis -> lista pojedynczych znaków
reguly = []                  # tu trafią reguły merge, w kolejności powstania

In [9]:
counts = {}
for i in range(len(tokeny) - 1):
    para = (tokeny[i], tokeny[i+1])
    counts[para] = counts.get(para, 0) + 1
print(para)

('3', '8')


In [10]:
with open("konstytucja.txt", encoding="utf-8") as f:
    konstytucja = f.read()
print("Korpus:", len(konstytucja), "znaków")

Korpus: 56535 znaków


In [11]:
print(konstytucja[:300])        # wycinek: pierwsze 300 znaków

Przyrodzona, niezbywalna i nienaruszalna godność człowieka stanowi fundament praw Rzeczypospolitej. Art. 2 Rzeczpospolita Polska jest państwem demokratycznym, troszczącym się o dobro wspólne, strzegącym niepodległości i nienaruszalności swojego terytorium oraz dziedzictwa narodowego. Art. 3 1. Władz


In [13]:
print("„" in konstytucja)      # `x in tekst` pyta "czy x występuje?" -> True/False; ma być False

False


In [15]:
konstytucja = konstytucja.replace("„", '"')
konstytucja = konstytucja.replace("”", '"')
konstytucja = konstytucja.replace("–", "-")
konstytucja = konstytucja.replace("\n", " ")
while "  " in konstytucja:
    konstytucja = konstytucja.replace("  ", " ")

In [18]:
counts = {}
for i in range(len(tokeny) - 1):
    para = (tokeny[i], tokeny[i+1])
    counts[para] = counts.get(para, 0) + 1

In [19]:
zwyciezca = max(counts, key=counts.get)

In [20]:
nowy_token = zwyciezca[0] + zwyciezca[1]
wynik = []
i = 0
while i < len(tokeny):
    if i < len(tokeny) - 1 and (tokeny[i], tokeny[i+1]) == zwyciezca:
        wynik.append(nowy_token)
        i = i + 2          # para zużyta -> przeskok o 2
    else:
        wynik.append(tokeny[i])
        i = i + 1          # zwykły token -> przeskok o 1
tokeny = wynik             # nowy korpus zastępuje stary

In [21]:
for runda in range(200):
    # ... 3b + 3c + 3d ...
    reguly.append(zwyciezca)

In [32]:
def encode(tekst, reguly):
    tokeny_nowe = list(tekst)
    for regula in reguly:              # reguły PO KOLEI, od najstarszej
        nowy_token = regula[0] + regula[1]
        # ... identyczny merge jak w 3d, tylko porównanie z `regula` ...
        tokeny_nowe = wynik            # wynik reguły N = wejście reguły N+1
    return tokeny_nowe

    nowy_tekst = "Rzeczpospolita Polska troszczy sie o dobro obywateli i zapewnia im wolnosc slowa."
    tokeny_nowe = encode(nowy_tekst, reguly)

In [36]:
# ============================================================
# MÓJ TOKENIZER BPE — wszystko w jednej komórce
# 1.wczytanie -> 2.normalizacja -> 3.trening -> 4.encoder -> 5.vocab -> 6.JSON
# ============================================================
import json

# ===== 1. WCZYTANIE =====
with open("konstytucja.txt", encoding="utf-8") as f:
    konstytucja = f.read()
print("Korpus:", len(konstytucja), "znaków")
print("Początek tekstu:", konstytucja[:200], "\n")

# ===== 2. NORMALIZACJA (znaki-bliźniaki; spacje ZOSTAJĄ) =====
konstytucja = konstytucja.replace("„", '"')
konstytucja = konstytucja.replace("”", '"')
konstytucja = konstytucja.replace("–", "-")
konstytucja = konstytucja.replace("\n", " ")
while "  " in konstytucja:
    konstytucja = konstytucja.replace("  ", " ")
print("Po normalizacji:", len(konstytucja), "znaków")
print("Czy zostały bliźniaki „ ?", "„" in konstytucja, "\n")

# ===== 3. TRENING (4.1 zliczanie + zwycięzca + 4.2 merge = 4.3 pętla) =====
tokeny = list(konstytucja)     # napis -> lista pojedynczych znaków
reguly = []                    # mózg tokenizera: reguły w kolejności powstania

for runda in range(200):
    # 4.1 — kreski przy parach
    counts = {}
    for i in range(len(tokeny) - 1):
        para = (tokeny[i], tokeny[i+1])
        counts[para] = counts.get(para, 0) + 1

    # zwycięzca rundy
    zwyciezca = max(counts, key=counts.get)

    # 4.2 — sklejanie (przeskok o 2 po parze, o 1 po zwykłym tokenie)
    nowy_token = zwyciezca[0] + zwyciezca[1]
    wynik = []
    i = 0
    while i < len(tokeny):
        if i < len(tokeny) - 1 and (tokeny[i], tokeny[i+1]) == zwyciezca:
            wynik.append(nowy_token)
            i = i + 2
        else:
            wynik.append(tokeny[i])
            i = i + 1
    tokeny = wynik             # nowy korpus zastępuje stary
    reguly.append(zwyciezca)

    # podgląd pierwszych 5 rund — jak na kartce
    if runda < 5:
        print("runda", runda + 1, "| zwycięzca:", zwyciezca, "| korpus ma", len(tokeny), "tokenów")

print("\nTrening zakończony. Liczba reguł:", len(reguly))
print("Pierwsze 10 reguł (parter):", reguly[:10])
print("Ostatnie 10 reguł (wysokie piętra):", reguly[-10:], "\n")

# ===== 4. ENCODER (4.4) — reguły PO KOLEI, bez liczenia =====
def encode(tekst, reguly):
    tokeny_nowe = list(tekst)
    for regula in reguly:
        nowy_token = regula[0] + regula[1]
        wynik = []
        i = 0
        while i < len(tokeny_nowe):
            if i < len(tokeny_nowe) - 1 and (tokeny_nowe[i], tokeny_nowe[i+1]) == regula:
                wynik.append(nowy_token)
                i = i + 2
            else:
                wynik.append(tokeny_nowe[i])
                i = i + 1
        tokeny_nowe = wynik
    return tokeny_nowe

nowy_tekst = "Rzeczpospolita Polska troszczy sie o dobro obywateli i zapewnia im wolnosc slowa."
tokeny_nowe = encode(nowy_tekst, reguly)   # UŻYCIE funkcji ("pieczenie ciasta")

print("Tokeny zdania testowego:", tokeny_nowe)
print("Znaki:", len(nowy_tekst), "| Tokeny:", len(tokeny_nowe))
print("Fertility (znaki na token):", round(len(nowy_tekst) / len(tokeny_nowe), 2), "\n")

# ===== 5. VOCAB: token -> ID =====
znaki_bazowe = sorted(set(konstytucja))    # alfabet bazowy, stały porządek
vocab = {}
for znak in znaki_bazowe:
    vocab[znak] = len(vocab)               # kolejne wolne numery: 0, 1, 2...
for regula in reguly:
    nowy_token = regula[0] + regula[1]
    if nowy_token not in vocab:
        vocab[nowy_token] = len(vocab)     # sklejenia wg kolejności powstania

tokeny_id = [vocab[t] for t in tokeny_nowe]
print("Rozmiar słownika (vocab):", len(vocab))
print("Tokeny jako ID:", tokeny_id, "\n")

# ===== 6. JSON — artefakt do oddania =====
wyniki = {
    "dataset": "Konstytucja RP (preambula + Rozdzial I i II)",
    "typ_tokenizera": "character-level BPE, czysty Python",
    "liczba_znakow_korpusu": len(konstytucja),
    "liczba_regul_merge": len(reguly),
    "rozmiar_slownika": len(vocab),
    "przykladowy_tekst": nowy_tekst,
    "tokeny": tokeny_nowe,
    "tokeny_id": tokeny_id,
    "liczba_tokenow": len(tokeny_nowe),
    "znaki_na_token": round(len(nowy_tekst) / len(tokeny_nowe), 3),
    "reguly_merge": [[a, b] for (a, b) in reguly],
    "vocab": vocab
}
with open("wyniki.json", "w", encoding="utf-8") as f:
    json.dump(wyniki, f, ensure_ascii=False, indent=2)
print("Zapisano wyniki.json — pobierz z panelu 📁 (trzy kropki -> Pobierz)")

Korpus: 56535 znaków
Początek tekstu: Przyrodzona, niezbywalna i nienaruszalna godność człowieka stanowi fundament praw Rzeczypospolitej. Art. 2 Rzeczpospolita Polska jest państwem demokratycznym, troszczącym się o dobro wspólne, strzegąc 

Po normalizacji: 56364 znaków
Czy zostały bliźniaki „ ? False 

runda 1 | zwycięzca: ('n', 'i') | korpus ma 55195 tokenów
runda 2 | zwycięzca: (' ', 'p') | korpus ma 54290 tokenów
runda 3 | zwycięzca: ('.', ' ') | korpus ma 53471 tokenów
runda 4 | zwycięzca: ('s', 't') | korpus ma 52719 tokenów
runda 5 | zwycięzca: ('a', ' ') | korpus ma 51978 tokenów

Trening zakończony. Liczba reguł: 200
Pierwsze 10 reguł (parter): [('n', 'i'), (' ', 'p'), ('.', ' '), ('s', 't'), ('a', ' '), ('e', ' '), ('z', 'e'), ('o', 'w'), (' ', 'w'), ('r', 'a')]
Ostatnie 10 reguł (wysokie piętra): [('z', 'w'), ('ie', 'r'), ('or', 'g'), ('ok', 're'), ('u', 'ch'), ('ł', 'os'), ('l', 'e'), ('a', 'r'), ('k', 'on'), ('m', 'ow')] 

Tokeny zdania testowego: ['Rze', 'cz', 'p', 'osp',